In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Creating few tools

In [3]:
@tool
def get_current_weather_tool(location: str) -> str:
    """
    Get the current weather in a given location
    """
    return f"The current weather in {location} is sunny with a temperature of 25°C."

@tool
def get_stock_price_tool(stock_symbol: str) -> str:
    """
    Get the current stock price of a given stock symbol
    """
    return f"The current stock price of {stock_symbol} is $150."

@tool
def get_news_tool(topic: str) -> str:
    """
    Get the latest news on a given topic
    """
    return f"The latest news on {topic} is that the stock market is up by 2% today."

### Binding tools to the model

In [ ]:
google_model_name = os.getenv("GOOGLE_MODEL")
google_model = init_chat_model(
    google_model_name,
    model_provider="google_genai"
)
google_model_with_tools = google_model.bind_tools([get_current_weather_tool, get_stock_price_tool, get_news_tool]);

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}}, output_version=None, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.5-flash-lite', temperature=None, client=<google.genai.client.Client o

### `invoke()` model with different queries

In [ ]:
tool_response = google_model_with_tools.invoke("What is the current weather in New York?")
print(tool_response.type) # ai

tool_calls = tool_response.tool_calls

if tool_calls:
    for tool_call in tool_calls:
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")
        # Invoke the ToolMessage to get the tool's response
else:
    print("No tool calls were made for this response.")

Tool: get_current_weather_tool
Args: {'location': 'New York'}


In [16]:
tool_response = google_model_with_tools.invoke("What is the current stock price of PYPL?")

tool_calls = tool_response.tool_calls

if tool_calls:
    for tool_call in tool_calls:
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")
else:
    print("No tool calls were made for this response.")

Tool: get_stock_price_tool
Args: {'stock_symbol': 'PYPL'}


In [17]:
tool_response = google_model_with_tools.invoke("Who won the IPL-2025?")

tool_calls = tool_response.tool_calls

if tool_calls:
    for tool_call in tool_calls:
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")
else:
    print("No tool calls were made for this response.")

Tool: get_news_tool
Args: {'topic': 'IPL 2025 winner'}


- Based on the `DocString` of the tool, the model will decide which tool to call and with what arguments. 
- The model will then call the tool and return the result.